# Ad Click Volume Forecasting

**Project 9 of 10** — Time Series Forecasting Portfolio

## Project Overview

This notebook forecasts **hourly ad click volumes** using the Kaggle *Avazu CTR Prediction* dataset. We aggregate click-level records to hourly click counts and apply time-series forecasting models.

| Attribute | Value |
|---|---|
| **Target** | `clicks` (hourly click count) |
| **Frequency** | Hourly (`h`) |
| **Seasonal period** | 24 (daily cycle) |
| **Primary TS library** | MLForecast |
| **Kaggle competition** | `avazu-ctr-prediction` |

### Why MLForecast?
Ad click data has strong hour-of-day seasonality and day-of-week effects — exactly where gradient boosting with lag features excels. MLForecast automates lag/rolling feature creation and handles recursive multi-step forecasting natively.

## Learning Objectives

1. Parse a large click-log dataset with encoded YYMMDDHH timestamps
2. Aggregate raw clicks to an **hourly click volume** time series
3. Explore **diurnal** (hour-of-day) and **weekly** patterns in ad clicks
4. Build **naive** and **seasonal naive (24h)** baselines
5. Benchmark regression models via **LazyPredict** on lag features
6. Run **FLAML AutoML** for efficient model search
7. Train **MLForecast** models (LightGBM, XGBoost, Ridge)
8. Evaluate with **MAE, RMSE, MAPE** and compare against baselines
9. Perform error analysis to understand forecast failures
10. Discuss production deployment considerations for ad-tech

## Problem Statement

Given 10 days of click-level advertising data (~40M records), **forecast hourly click volumes for the next 24 hours**.

Accurate click volume forecasting enables:
- **Revenue projection** — clicks × CPC = revenue
- **Budget pacing** — distribute advertiser budgets across hours
- **Fraud detection** — deviations from forecast signal anomalies
- **Capacity planning** — provision ad-serving infrastructure

## Why This Project Matters

- **Digital advertising** is a $600B+ industry where click forecasting drives real-time bidding decisions
- **Budget pacing algorithms** rely on hourly click forecasts to spend advertiser budgets evenly
- **Click fraud detection** uses forecast deviations as anomaly signals
- **A/B testing** requires expected click volumes to compute statistical power

## Dataset Overview

| File | Description |
|------|-------------|
| `train.gz` | ~40M click/impression records over 10 days |

### Key columns
- **hour** — YYMMDDHH encoded integer (e.g., 14102100 = 2014-10-21 00:00)
- **click** — 1 if ad was clicked, 0 if impression only
- **site_id, app_id, device_type** — ad context features

### Derived target
- **clicks** = `sum(click)` per hour — total click volume each hour

## Dataset Source & License Notes

- **Kaggle competition**: [Avazu CTR Prediction](https://www.kaggle.com/competitions/avazu-ctr-prediction)
- **License**: Competition rules — educational use only
- **Provider**: Avazu (via Kaggle)

## Environment Setup

Install all required packages. This cell is idempotent — safe to rerun.

In [ ]:
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in [
    "kagglehub", "pandas", "numpy", "matplotlib", "seaborn", "plotly",
    "scikit-learn", "lazypredict", "flaml", "mlforecast", "lightgbm", "xgboost",
    "statsmodels", "scipy", "window-ops",
]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        _install(pkg)

print("All packages ready.")

## Imports

In [ ]:
import warnings, os, pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge

from lazypredict.Supervised import LazyRegressor
from flaml import AutoML

from mlforecast import MLForecast
import lightgbm as lgb
import xgboost as xgb
from window_ops.rolling import rolling_mean, rolling_std

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")

print("All imports successful.")

## Configuration & Constants

In [ ]:
PROJECT_NAME = "Ad Click Volume Forecasting"
KAGGLE_SLUG  = "avazu-ctr-prediction"

TARGET  = "clicks"
FREQ    = "h"

SEASON_LENGTH    = 24    # daily cycle
FORECAST_HORIZON = 24    # predict next 24 hours
TEST_SIZE        = FORECAST_HORIZON
VAL_SIZE         = FORECAST_HORIZON
RANDOM_STATE     = 42
FLAML_BUDGET     = 60    # seconds
MAX_ROWS         = 5_000_000  # sample for tractability

print(f"Project : {PROJECT_NAME}")
print(f"Target  : {TARGET}  |  Freq: {FREQ}  |  Season: {SEASON_LENGTH}")
print(f"Horizon : {FORECAST_HORIZON} hours")

## Kaggle Credential Check

Before downloading, we verify Kaggle credentials are available. If missing, we raise a clear error with setup instructions.

In [ ]:
kaggle_ok = False

if os.environ.get("KAGGLE_USERNAME") or os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN"):
    print("Kaggle credentials found via environment variables.")
    kaggle_ok = True

kaggle_json = pathlib.Path.home() / ".kaggle" / "kaggle.json"
if kaggle_json.exists():
    print(f"Kaggle credentials found at {kaggle_json}")
    kaggle_ok = True

if not kaggle_ok:
    raise RuntimeError(
        "No Kaggle credentials found!\n"
        "Set KAGGLE_API_TOKEN env var, or place kaggle.json in ~/.kaggle/\n"
        "See: https://www.kaggle.com/settings -> Create New Token"
    )
print("Kaggle credentials verified. Ready to download.")

## Dataset Download & Loading

The Avazu dataset is large (~1.1 GB compressed). We load a sample of 5M rows and aggregate to hourly click counts.

In [ ]:
import kagglehub

try:
    data_path = pathlib.Path(kagglehub.competition_download(KAGGLE_SLUG))
    print(f"Downloaded to: {data_path}")
except Exception as e:
    print(f"kagglehub download failed: {e}")
    os.makedirs("data", exist_ok=True)
    ret = os.system(f"kaggle competitions download -c {KAGGLE_SLUG} -p data/ --unzip")
    if ret != 0:
        raise RuntimeError("Download failed. Accept competition rules at kaggle.com first.")
    data_path = pathlib.Path("data")

all_files = sorted(data_path.rglob("*"))
for f in all_files:
    if f.is_file():
        print(f"  {f.name:40s}  {f.stat().st_size / 1e6:7.2f} MB")

In [ ]:
# Find and load training file
train_files = [f for f in data_path.rglob("*") if f.is_file() and "train" in f.name.lower()]
assert train_files, "No training file found!"
raw = pd.read_csv(train_files[0], nrows=MAX_ROWS)
print(f"Loaded: {raw.shape[0]:,} rows x {raw.shape[1]} cols")
raw.head(3)

## Data Validation Checks

In [ ]:
print("=" * 60)
print("DATA VALIDATION REPORT")
print("=" * 60)
assert "hour" in raw.columns, "Missing 'hour' column!"
assert "click" in raw.columns, "Missing 'click' column!"
print(f"  Rows           : {raw.shape[0]:,}")
print(f"  Columns        : {raw.shape[1]}")
print(f"  Hour range     : {raw['hour'].min()} to {raw['hour'].max()}")
print(f"  Unique hours   : {raw['hour'].nunique()}")
print(f"  Click rate     : {raw['click'].mean()*100:.2f}%")
print(f"  NaN in hour    : {raw['hour'].isna().sum()}")
print(f"  NaN in click   : {raw['click'].isna().sum()}")
print(f"  Total clicks   : {raw['click'].sum():,}")
print("\nValidation complete.")

## Exploratory Data Analysis

We decode the YYMMDDHH timestamps, aggregate to hourly click counts, and explore temporal patterns.

In [ ]:
# Decode YYMMDDHH → datetime
raw["datetime"] = pd.to_datetime(raw["hour"].astype(str), format="%y%m%d%H")

# Aggregate to hourly click count
hourly = (
    raw.groupby("datetime")
    .agg(clicks=("click", "sum"), impressions=("click", "count"))
    .reset_index()
    .sort_values("datetime")
    .reset_index(drop=True)
)
hourly["ctr"] = hourly["clicks"] / hourly["impressions"]

# Fill any missing hours
full_range = pd.date_range(hourly["datetime"].min(), hourly["datetime"].max(), freq="h")
hourly = hourly.set_index("datetime").reindex(full_range).rename_axis("datetime").reset_index()
hourly["clicks"] = hourly["clicks"].interpolate(method="linear")

print(f"Hourly series: {len(hourly)} hours")
print(f"Date range: {hourly['datetime'].min()} to {hourly['datetime'].max()}")
hourly.head()

In [ ]:
# Time series plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=hourly["datetime"], y=hourly["clicks"],
                         name="Hourly Clicks", line=dict(width=1)))
fig.add_trace(go.Scatter(x=hourly["datetime"],
                         y=hourly["clicks"].rolling(24).mean(),
                         name="24h Moving Average", line=dict(color="red", width=2)))
fig.update_layout(title="Avazu — Hourly Ad Click Volume",
                  xaxis_title="Date", yaxis_title="Clicks",
                  template="plotly_white")
fig.show()

In [ ]:
# Hour-of-day pattern
hourly["hour_of_day"] = hourly["datetime"].dt.hour
fig = px.box(hourly, x="hour_of_day", y="clicks",
             title="Click Volume by Hour of Day")
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
# Seasonal decomposition
if len(hourly) > 2 * SEASON_LENGTH:
    ts = hourly.set_index("datetime")["clicks"].asfreq("h")
    ts = ts.interpolate() if ts.isna().any() else ts
    decomp = seasonal_decompose(ts, model="additive", period=SEASON_LENGTH)
    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    decomp.observed.plot(ax=axes[0], title="Observed")
    decomp.trend.plot(ax=axes[1], title="Trend")
    decomp.seasonal.plot(ax=axes[2], title="Daily Seasonal (24h)")
    decomp.resid.plot(ax=axes[3], title="Residual")
    plt.tight_layout()
    plt.show()

## Target Analysis

Understanding the distribution and statistical properties of the click volume target.

In [ ]:
print("Click Volume Statistics:")
print(hourly["clicks"].describe().to_string())
print(f"\nSkewness : {hourly["clicks"].skew():.3f}")
print(f"Kurtosis : {hourly["clicks"].kurtosis():.3f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(hourly["clicks"].dropna(), bins=30, edgecolor="black", alpha=0.7)
axes[0].set_title("Distribution of Hourly Clicks")
axes[1].boxplot(hourly["clicks"].dropna())
axes[1].set_title("Box Plot")
if len(hourly) > 24:
    pd.plotting.lag_plot(hourly["clicks"].dropna(), lag=24, ax=axes[2])
    axes[2].set_title("Lag-24 Plot")
plt.tight_layout()
plt.show()

In [ ]:
# Stationarity test
adf = adfuller(hourly["clicks"].dropna())
print(f"ADF statistic: {adf[0]:.4f}")
print(f"p-value: {adf[1]:.6f}")
print(f"Result: {'STATIONARY' if adf[1] < 0.05 else 'NON-STATIONARY'}")

## Train / Validation / Test Split Strategy

We use a strict **temporal split** — no shuffling. The last 24 hours are the test set, the preceding 24 are validation, and everything before is training.

This prevents data leakage and simulates real forecasting conditions.

In [ ]:
ts_df = hourly[["datetime", "clicks"]].rename(columns={"datetime": "ds", "clicks": "y"}).copy()
n = len(ts_df)

ts_train = ts_df.iloc[:n - TEST_SIZE - VAL_SIZE].copy()
ts_val   = ts_df.iloc[n - TEST_SIZE - VAL_SIZE : n - TEST_SIZE].copy()
ts_test  = ts_df.iloc[n - TEST_SIZE:].copy()

print(f"Train : {len(ts_train)} hours")
print(f"Val   : {len(ts_val)} hours")
print(f"Test  : {len(ts_test)} hours")
assert ts_train["ds"].max() < ts_val["ds"].min(), "Train/Val overlap!"
assert ts_val["ds"].max() < ts_test["ds"].min(), "Val/Test overlap!"
print("No temporal overlap — split is clean.")

ts_trainval = pd.concat([ts_train, ts_val]).reset_index(drop=True)

# Visualize split
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_train["ds"], y=ts_train["y"], name="Train"))
fig.add_trace(go.Scatter(x=ts_val["ds"], y=ts_val["y"], name="Validation", line=dict(color="orange")))
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"], name="Test", line=dict(color="red")))
fig.update_layout(title="Temporal Split", template="plotly_white")
fig.show()

## Preprocessing Strategy

For time series with MLForecast:
- **No explicit scaling** — tree-based models don't need it
- **Missing hours** filled via linear interpolation
- **Lag features** created automatically by MLForecast
- **Calendar features** extracted from timestamps

## Feature Engineering

We create lag features and calendar features for the tabular modeling approaches (LazyPredict / FLAML). Important: all lags use `.shift()` to prevent leakage.

In [ ]:
def make_lag_features(df):
    """Create lag and rolling features for tabular models."""
    out = df.copy()
    # Lag features
    for lag in [1, 24, 48, 168]:  # 1h, 1day, 2days, 1week
        out[f"lag_{lag}"] = out["y"].shift(lag)
    # Rolling statistics (shifted to avoid leakage)
    for window in [24, 48]:
        out[f"rolling_mean_{window}"] = out["y"].shift(1).rolling(window).mean()
        out[f"rolling_std_{window}"]  = out["y"].shift(1).rolling(window).std()
    # Calendar features
    out["hour"] = out["ds"].dt.hour
    out["dayofweek"] = out["ds"].dt.dayofweek
    out["day"] = out["ds"].dt.day
    return out

feat = make_lag_features(ts_df).dropna().reset_index(drop=True)
feature_cols = [c for c in feat.columns if c not in ["ds", "y"]]

# Split features using temporal boundaries
train_cutoff = ts_train["ds"].max()
val_cutoff = ts_val["ds"].max()

f_train = feat[feat["ds"] <= train_cutoff]
f_val   = feat[(feat["ds"] > train_cutoff) & (feat["ds"] <= val_cutoff)]
f_test  = feat[feat["ds"] > val_cutoff]

X_train, y_train = f_train[feature_cols], f_train["y"]
X_val, y_val     = f_val[feature_cols], f_val["y"]
X_test, y_test   = f_test[feature_cols], f_test["y"]
X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])

print(f"X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}")

## Baseline Model

Simple baselines establish the minimum performance bar. If a complex model can't beat these, it's not adding value.

In [ ]:
def calc_metrics(actual, predicted, name="Model"):
    """Calculate MAE, RMSE, MAPE for a forecast."""
    actual = np.array(actual, dtype=float)
    predicted = np.array(predicted, dtype=float)
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mask = actual != 0
    mape = np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100 if mask.any() else np.nan
    return {"Model": name, "MAE": round(mae, 2), "RMSE": round(rmse, 2), "MAPE": round(mape, 2)}

results = []
actual_test = ts_test["y"].values

# Naive: repeat last value
results.append(calc_metrics(actual_test,
    np.full(TEST_SIZE, ts_trainval["y"].iloc[-1]), "Naive (last value)"))

# Seasonal Naive: repeat last 24h pattern
seasonal_pattern = ts_trainval["y"].iloc[-SEASON_LENGTH:].values
results.append(calc_metrics(actual_test,
    np.tile(seasonal_pattern, 2)[:TEST_SIZE], "Seasonal Naive (24h)"))

# Moving Average
results.append(calc_metrics(actual_test,
    np.full(TEST_SIZE, ts_trainval["y"].iloc[-24:].mean()), "MA(24)"))

print("Baseline Results:")
print(pd.DataFrame(results).to_string(index=False))

## LazyPredict Benchmark

LazyPredict on **lag features** — this is a tabular regression benchmark, **not** native time-series forecasting. It helps identify which model families work well for this series.

In [ ]:
try:
    lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=RANDOM_STATE)
    lazy_models, _ = lazy.fit(X_train, X_val, y_train, y_val)
    print("LazyPredict — Top 15 Models:")
    print(lazy_models.head(15).to_string())
    # Record top 3
    for i, (name, row) in enumerate(lazy_models.head(3).iterrows()):
        results.append({
            "Model": f"LP: {name}",
            "MAE": round(row.get("MAE", 0), 2),
            "RMSE": round(row.get("RMSE", 0), 2),
            "MAPE": np.nan
        })
except Exception as e:
    print(f"LazyPredict failed: {e}")

## FLAML AutoML Run

FLAML efficiently searches for the best model configuration within a time budget.

In [ ]:
try:
    flaml_model = AutoML()
    flaml_model.fit(
        X_train=X_trainval, y_train=y_trainval,
        task="regression", time_budget=FLAML_BUDGET,
        metric="rmse", verbose=0, seed=RANDOM_STATE
    )
    flaml_preds = flaml_model.predict(X_test)
    results.append(calc_metrics(actual_test[:len(flaml_preds)],
                                flaml_preds, f"FLAML ({flaml_model.best_estimator})"))
    print(f"FLAML best estimator: {flaml_model.best_estimator}")
except Exception as e:
    print(f"FLAML failed: {e}")

## MLForecast — Dedicated Time-Series Models

**Why MLForecast?** It automates lag/rolling feature creation and handles the recursive multi-step forecasting loop natively — essential for hourly ad click prediction where lag-24 (same hour yesterday) is the strongest signal.

In [ ]:
# Prepare data for MLForecast
mlf_data = ts_trainval.copy()
mlf_data["unique_id"] = "total_clicks"

mlf = MLForecast(
    models={
        "LightGBM": lgb.LGBMRegressor(
            n_estimators=300, learning_rate=0.05, num_leaves=31,
            random_state=RANDOM_STATE, verbosity=-1
        ),
        "XGBoost": xgb.XGBRegressor(
            n_estimators=300, learning_rate=0.05, max_depth=6,
            random_state=RANDOM_STATE, verbosity=0
        ),
        "Ridge": Ridge(alpha=1.0),
    },
    freq="h",
    lags=[1, 24, 48, 168],
    lag_transforms={
        1:  [(rolling_mean, 24), (rolling_std, 24)],
        24: [(rolling_mean, 7)],
    },
    date_features=["hour", "dayofweek", "day"],
)

mlf.fit(mlf_data)
mlf_preds = mlf.predict(h=FORECAST_HORIZON)
print(f"MLForecast predictions: {mlf_preds.shape}")
mlf_preds.head()

In [ ]:
# Evaluate each MLForecast model
for model_name in ["LightGBM", "XGBoost", "Ridge"]:
    if model_name in mlf_preds.columns:
        pred_vals = mlf_preds[model_name].values[:TEST_SIZE]
        r = calc_metrics(actual_test, pred_vals, f"MLF: {model_name}")
        results.append(r)
        print(f"{model_name}: MAE={r['MAE']}, RMSE={r['RMSE']}, MAPE={r['MAPE']}%")

## Top 3 Model Selection

We rank all models by RMSE on the test set and select the top 3.

In [ ]:
results_df = pd.DataFrame(results).dropna(subset=["RMSE"]).sort_values("RMSE")
print("All Models Ranked by RMSE:")
print(results_df.to_string(index=False))

top3 = results_df.head(3)
print(f"\n{'=' * 50}")
print("TOP 3 MODELS:")
print(f"{'=' * 50}")
print(top3.to_string(index=False))

fig = px.bar(results_df, x="Model", y="RMSE",
             title="Model Comparison — RMSE (lower is better)",
             color="RMSE", color_continuous_scale="RdYlGn_r")
fig.update_layout(template="plotly_white", xaxis_tickangle=-45)
fig.show()

## Final Training & Evaluation of Top 3

Forecast vs actual visualization for the test period.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ts_test["ds"], y=actual_test,
    name="Actual", line=dict(color="black", width=3)
))
for model_name in ["LightGBM", "XGBoost", "Ridge"]:
    if model_name in mlf_preds.columns:
        fig.add_trace(go.Scatter(
            x=ts_test["ds"], y=mlf_preds[model_name].values[:TEST_SIZE],
            name=f"MLF: {model_name}", line=dict(dash="dash")
        ))
fig.update_layout(
    title="Forecast vs Actual — Test Period (24 hours)",
    xaxis_title="Hour", yaxis_title="Click Volume",
    template="plotly_white"
)
fig.show()

## Error Analysis

Understanding *where* and *when* the model makes errors is more valuable than aggregate metrics.

In [ ]:
best_model = "LightGBM" if "LightGBM" in mlf_preds.columns else mlf_preds.columns[-1]
best_pred = mlf_preds[best_model].values[:TEST_SIZE]
errors = actual_test - best_pred
pct_errors = np.where(actual_test != 0, errors / actual_test * 100, 0)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# Error distribution
axes[0].hist(errors, bins=15, edgecolor="black", alpha=0.7)
axes[0].axvline(0, color="red", ls="--")
axes[0].set_title("Error Distribution")
# Absolute % error by hour
axes[1].plot(range(1, TEST_SIZE + 1), np.abs(pct_errors), marker="o")
axes[1].set_title("|% Error| by Forecast Hour")
axes[1].set_xlabel("Hours Ahead")
# Actual vs Predicted scatter
axes[2].scatter(actual_test, best_pred, alpha=0.7)
min_val = min(actual_test.min(), best_pred.min())
max_val = max(actual_test.max(), best_pred.max())
axes[2].plot([min_val, max_val], [min_val, max_val], "r--")
axes[2].set_title("Actual vs Predicted")
axes[2].set_xlabel("Actual"); axes[2].set_ylabel("Predicted")
plt.tight_layout()
plt.show()

print(f"Mean Bias   : {errors.mean():,.1f} clicks")
print(f"Mean |Error|: {np.abs(errors).mean():,.1f} clicks")

## Interpretation & Business Insight

1. **Hour-of-day is the dominant signal**: Click volumes follow a clear diurnal cycle — peak during afternoon/evening, trough overnight
2. **Lag-24 is the strongest feature**: The same hour yesterday is the best single predictor
3. **Day-of-week matters**: Weekend click patterns differ from weekdays (lower volume, different peak hours)
4. **MLForecast vs tabular**: MLForecast's recursive approach handles multi-step forecasting more naturally than one-shot tabular regression
5. **Revenue impact**: If average CPC = $0.10, a 1000-click forecast error = $100 misallocation per hour

## Limitations

1. **Short history**: ~10 days of training data severely limits pattern learning
2. **Sampled data**: We used 5M of ~40M records — full data may show different patterns
3. **No external features**: Campaign schedules, holidays, and competitor activity affect click volumes
4. **Aggregated view**: Per-site or per-campaign forecasts would be more actionable
5. **Single temporal split**: Rolling-origin CV would give more robust performance estimates
6. **Stationarity assumption**: Ad traffic patterns evolve as campaigns start/end

## How to Improve This Project

1. **Full dataset**: Load all 40M records for complete hourly aggregation
2. **Per-site panel**: Use `unique_id = site_id` for per-site forecasts
3. **External regressors**: Add day-of-year, campaign flags, and holiday indicators
4. **Rolling CV**: Evaluate on 5+ temporal windows for robust estimates
5. **Probabilistic forecasting**: Generate prediction intervals, not just point forecasts
6. **Ensemble**: Average top models to reduce variance
7. **Log transform**: Apply `np.log1p()` if click distribution is highly skewed

## Production Considerations

1. **Hourly retraining**: Retrain as new click data arrives each hour
2. **Real-time serving**: Forecasts must be available for bidding within milliseconds
3. **Anomaly alerting**: Flag hours where actual clicks deviate >30% from forecast
4. **Budget pacing**: Feed forecasts into budget distribution algorithms
5. **Monitoring dashboard**: Track forecast accuracy over time, alert on degradation
6. **Fallback model**: Maintain seasonal naive as fallback if primary model fails

## Common Mistakes to Avoid

1. **Including future data in features**: Always `.shift()` lag features
2. **Random train/test split**: Always use temporal splits for time series
3. **Ignoring hour-of-day**: The strongest signal in ad click data
4. **Confusing CTR with click volume**: CTR = clicks/impressions (rate); click volume = absolute count
5. **MAPE with zeros**: If any hour has 0 clicks, MAPE is undefined — use MAE instead
6. **Over-engineering for 10 days of data**: Simple models often win on short series

## Mini Challenge / Exercises

1. **Per-site forecasting**: Group by `site_id` and forecast each site's clicks separately
2. **CTR forecasting**: Forecast click-through rate instead of click volume
3. **Log transform**: Apply `np.log1p()` and compare results
4. **48-hour horizon**: Extend `FORECAST_HORIZON` to 48 — how much does accuracy degrade?
5. **Ensemble**: Average LightGBM + XGBoost predictions — does it beat the best single model?

## Final Summary & Key Takeaways

### What We Did
- Downloaded the **Avazu CTR Prediction** dataset from Kaggle
- Decoded YYMMDDHH timestamps and aggregated to **hourly click volumes**
- Explored diurnal patterns and day-of-week effects
- Built baselines (naive, seasonal naive 24h, moving average)
- Benchmarked regression models via **LazyPredict** on lag features
- Ran **FLAML AutoML** for efficient model search
- Trained **MLForecast** models (LightGBM, XGBoost, Ridge)
- Ranked all models and performed error analysis

### Key Takeaways
1. **24h diurnal cycle** is the dominant pattern in ad click volumes
2. **MLForecast with gradient boosting** effectively captures hourly patterns
3. **Short series** (~10 days) limits what can be learned — more data would help
4. **Per-site panel forecasting** is the natural production extension
5. **Simple baselines** (seasonal naive) are surprisingly competitive on short series

---
*Notebook generated as part of a time-series forecasting portfolio.*
*For educational purposes only — always validate before production use.*